# Project 11: Analisis A/B test iklan vs PSA

Notebook ini menjawab Q1–Q14 dari rancangan dengan dataset `marketing_AB.csv`. Angka dihitung ulang dengan modul `code/2-analysis/python/abtest.py` dan dicocokkan dengan `results/`.

Aturan uji ditetapkan sebelum melihat hasil: alpha 0,05 dua arah, power 80%, MDE 0,5 poin.

## Q1 · Kualitas data
Grup valid, tidak ada user ganda, tidak ada nilai hilang.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '../../code/2-analysis/python')
from abtest import two_prop_test, srm_test, sample_size_per_group, cohens_h
raw = pd.read_csv('../../data/raw/marketing_AB.csv', index_col=0)
print('baris:', len(raw), '| grup:', raw['test group'].value_counts().to_dict())
print('user ganda:', raw['user id'].duplicated().sum(), '| nilai hilang:', int(raw.isna().sum().sum()))
print(pd.read_csv('../1-data-preparation/cleaning_log.csv').to_string(index=False))

baris: 588101 | grup: {'ad': 564577, 'psa': 23524}
user ganda: 0 | nilai hilang: 0
                                step  rows_in  rows_out               detail
                                baca   588101    588101 kolom indeks dibuang
                    grup tidak valid   588101    588101   grup selain ad/psa
user ganda (ambil kunjungan pertama)   588101    588101     user_id duplikat


## Q2 · SRM (pembagian grup 50:50)
Chi-square terhadap rasio rencana. Jika gagal, kesimpulan kausal perlu batasan.

In [2]:
g = raw.groupby('test group')['converted'].agg(['count', 'sum'])
n_ad, n_psa = int(g.loc['ad', 'count']), int(g.loc['psa', 'count'])
srm = srm_test(n_ad, n_psa)
print(f"ad {n_ad} ({n_ad/(n_ad+n_psa):.1%}), psa {n_psa} ({n_psa/(n_ad+n_psa):.1%})")
print('chi2 =', round(srm['chi2'], 1), '| p =', srm['p_value'], '| SRM gagal:', srm['srm'])

ad 564577 (96.0%), psa 23524 (4.0%)
chi2 = 497768.8 | p = 0.0 | SRM gagal: True


## Q4–Q6 · Conversion rate, uji z, dan CI
Uji z dua proporsi dan CI selisih 95%.

In [3]:
x_ad, x_psa = int(g.loc['ad', 'sum']), int(g.loc['psa', 'sum'])
r = two_prop_test(x_ad, n_ad, x_psa, n_psa)
print(f"CR ad {r['cr_a']:.2%}, CR psa {r['cr_b']:.2%}")
print(f"selisih {r['diff']*100:.2f} poin, CI 95% {r['ci_low']*100:.2f} s.d. {r['ci_high']*100:.2f}")
print(f"lift relatif {r['rel_lift']:.1%}, z = {r['z']:.2f}, p = {r['p_value']:.2e}")

CR ad 2.55%, CR psa 1.79%
selisih 0.77 poin, CI 95% 0.59 s.d. 0.94
lift relatif 43.1%, z = 7.37, p = 1.71e-13


## Q7 · Cohen's h dan power
Cohen's h kecil. Daya untuk MDE 0,5 poin dari baseline PSA dihitung dengan ukuran sampel.

In [4]:
print('Cohen h =', round(r['cohens_h'], 4))
print('ukuran sampel per grup untuk MDE 0,5 poin:', sample_size_per_group(r['cr_b'], 0.005))

Cohen h = 0.053
ukuran sampel per grup untuk MDE 0,5 poin: 6237


## Q8–Q9 · Segmen hari dan intensitas iklan
Segmen hari memakai `most ads day`. Kuartil memakai `total ads`.

In [5]:
import numpy as np
day = raw.assign(grp=raw['test group']).groupby(['most ads day', 'test group'])['converted'].agg(['count', 'sum']).unstack()
rows = []
for d in day.index:
    a = two_prop_test(int(day.loc[d, ('sum','ad')]), int(day.loc[d, ('count','ad')]), int(day.loc[d, ('sum','psa')]), int(day.loc[d, ('count','psa')]))
    rows.append((d, round(a['diff']*100, 2), f"{a['p_value']:.2e}"))
pd.DataFrame(rows, columns=['hari', 'selisih (poin)', 'p (sebelum Bonferroni)'])

,hari,selisih (poin),p (sebelum Bonferroni)
0,Friday,0.62,1.16e-02
1,Monday,1.07,5.08e-04
2,Saturday,0.73,7.48e-03
3,Sunday,0.40,1.57e-01
4,Thursday,0.14,5.55e-01
5,Tuesday,1.60,6.63e-07
6,Wednesday,0.96,3.76e-04


In [6]:
q = raw.assign(kuartil=pd.qcut(raw['total ads'].rank(method='first'), 4, labels=['Q1','Q2','Q3','Q4']))
rows = []
for k, sub in q.groupby('kuartil', observed=True):
    gg = sub.groupby('test group')['converted'].agg(['count', 'sum'])
    a = two_prop_test(int(gg.loc['ad','sum']), int(gg.loc['ad','count']), int(gg.loc['psa','sum']), int(gg.loc['psa','count']))
    rows.append((k, round(a['diff']*100, 2), f"{a['p_value']:.2e}"))
pd.DataFrame(rows, columns=['kuartil total ads', 'selisih (poin)', 'p'])

,kuartil total ads,selisih (poin),p
0,Q1,0.04,5.69e-01
1,Q2,-0.09,3.22e-01
2,Q3,-0.11,4.52e-01
3,Q4,3.01,4.02e-16


## Q12–Q13 · Bootstrap dan sensitivitas
Bootstrap 2.000 resample. Sensitivitas: buang user dengan total ads di atas P99.

In [7]:
from abtest import bootstrap_diff
print('bootstrap:', bootstrap_diff(x_ad, n_ad, x_psa, n_psa, reps=2000))
p99 = raw['total ads'].quantile(0.99)
sub = raw[raw['total ads'] <= p99]
gs = sub.groupby('test group')['converted'].agg(['count','sum'])
s = two_prop_test(int(gs.loc['ad','sum']), int(gs.loc['ad','count']), int(gs.loc['psa','sum']), int(gs.loc['psa','count']))
print(f'P99 = {p99}; selisih tanpa outlier = {s["diff"]*100:.2f} poin, p = {s["p_value"]:.2e}')

bootstrap: {'ci_low': 0.005928253260851024, 'ci_high': 0.009520550235063088}
P99 = 202.0; selisih tanpa outlier = 0.79 poin, p = 9.54e-15


## Q10 · Estimasi dampak (estimated)
Tambahan konversi per 1 juta user. Estimasi per bulan tidak bisa dihitung karena tidak ada data trafik per bulan.

In [8]:
print(f"estimated: {r['diff']*1e6:,.0f} konversi per 1 juta user (CI {r['ci_low']*1e6:,.0f} s.d. {r['ci_high']*1e6:,.0f})".replace(',', '.'))

estimated: 7.692 konversi per 1 juta user (CI 5.873 s.d. 9.360)


## Q14 · Cek silang SQL vs Python
Hasil DuckDB dibandingkan dengan Python. Tabel lengkap di `results/2-analysis/crosscheck.csv`.

In [9]:
import duckdb
f = pd.read_csv('../2-analysis/crosscheck.csv')
print(f['hasil'].value_counts().to_dict())
con = duckdb.connect('../../abtest.duckdb', read_only=True)
print(con.sql('SELECT * FROM v_group_summary').df())

{'PASS': 36}
   grp       n  konversi  conv_rate  median_total_ads  mean_total_ads
0   ad  564577   14423.0   0.025547              13.0       24.823365
1  psa   23524     420.0   0.017854              12.0       24.761138


## Kesimpulan
- Iklan menghasilkan conversion rate lebih tinggi dibanding PSA (selisih 0,77 poin, signifikan).
- Pembagian grup gagal SRM (96% vs 4%). Kesimpulan kausal perlu batasan.
- Efek hanya terlihat di kuartil total ads teratas.
- Belum rollout berdasarkan angka ini. Pastikan alokasi grup, lalu uji ulang dengan 50:50.

Lihat [Report](../../design-and-report/index.html) dan [README](../../README.md) untuk ringkasan lengkap.